In [2]:
#TASK 1 - LOAD THE DATA
import numpy as np
from PIL import Image
import os

DATASET_PATH = "dataset"

In [3]:
# Names of 10 character classes

class_names = [
    "bart_simpson",
    "charles_montgomery_burns",
    "homer_simpson",
    "krusty_the_clown",
    "lisa_simpson",
    "marge_simpson",
    "milhouse_van_houten",
    "moe_szyslak",
    "ned_flanders",
    "principal_skinner"
]


In [4]:
def load_images(data_dir, image_mode):
    """Load JPEG images from class subfolders into flattened, normalised
    NumPy arrays, with integer labels based on folder order."""
    images = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)

        for filename in sorted(os.listdir(class_dir)):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                image_path = os.path.join(class_dir, filename)
                with Image.open(image_path) as image:
                    image = image.convert(image_mode)
                    image_array = np.asarray(image, dtype=np.float32).flatten() / 255.0
                images.append(image_array)
                labels.append(label)
    X = np.array(images)
    y = np.array(labels)
    return X, y

In [5]:
#Load grayscale training and test data
X_train_gray, y_train_gray = load_images(
    os.path.join(DATASET_PATH, "grayscale", "train"), "L"
)
X_test_gray, y_test_gray = load_images(
    os.path.join(DATASET_PATH, "grayscale", "test"), "L"
)

#Load RGB training and test data
X_train_rgb, y_train_rgb = load_images(
    os.path.join(DATASET_PATH, "rgb", "train"), "RGB"
)
X_test_rgb, y_test_rgb = load_images(
    os.path.join(DATASET_PATH, "rgb", "test"), "RGB"
)

In [6]:
# Splitin off a validation set from the training data
np.random.seed(1234)
valid_size = X_test_gray.shape[0]
indices = np.random.choice(X_train_gray.shape[0], valid_size, replace=False)

X_valid_gray = X_train_gray[indices]
y_valid_gray = y_train_gray[indices]
X_train_gray = np.delete(X_train_gray, indices, axis=0)
y_train_gray = np.delete(y_train_gray, indices, axis=0)

X_valid_rgb = X_train_rgb[indices]
y_valid_rgb = y_train_rgb[indices]
X_train_rgb = np.delete(X_train_rgb, indices, axis=0)
y_train_rgb = np.delete(y_train_rgb, indices, axis=0)


In [7]:
# Checking size of each dataset
print("Grayscale:", X_train_gray.shape, X_valid_gray.shape, X_test_gray.shape)
print("RGB:      ", X_train_rgb.shape, X_valid_rgb.shape, X_test_rgb.shape)

Grayscale: (6000, 784) (2000, 784) (2000, 784)
RGB:       (6000, 2352) (2000, 2352) (2000, 2352)


In [8]:
# TASK 2

class BinaryPerceptron:
 
    def __init__(self, n_inputs, alpha=0.01):
        self.weights = np.full(n_inputs, 0.1)
        self.bias = 0.1
        self.alpha = alpha
 
    def net_input(self, x):
        return np.dot(x, self.weights) + self.bias
 
    def predict(self, x):
        return 1 if self.net_input(x) >= 0 else 0
 
    def apply_learning_rule(self, x, y):
        # Applying the perceptron update rule: comparing my prediction to the true label and only adjusting weights/bias when they differ
        g = self.predict(x)
        self.weights = self.weights + self.alpha * (y - g) * x
        self.bias = self.bias + self.alpha * (y - g)

In [9]:

def train(model, X_train, y_train, X_valid, y_valid, epochs=10):
 
    for epoch in range(epochs):
 
        # Shuffling the training data each epoch so I'm not feeding it examples in the same class-grouped order every time
        indices = np.random.permutation(len(X_train))
 
        for i in indices:
            model.apply_learning_rule(X_train[i], y_train[i])
 
        correct = 0
 
        for i in range(len(X_valid)):
            if model.predict(X_valid[i]) == y_valid[i]:
                correct += 1
 
        accuracy = correct / len(X_valid)
 
        print(f"Epoch {epoch + 1} - Validation accuracy: {accuracy:.2f}")

In [18]:
class MultiClassPerceptron:
 
    def __init__(self, n_inputs, n_classes=10, alpha=0.01):
        self.perceptrons = []
 
        for i in range(n_classes):
            self.perceptrons.append(
                BinaryPerceptron(n_inputs, alpha)
            )
 
    def predict(self, x):
        # Getting a raw score from each of the 10 perceptrons, then picking whichever class's perceptron is most confident
        scores = []
 
        for perceptron in self.perceptrons:
            scores.append(perceptron.net_input(x))
 
        return np.argmax(scores)
 
    def apply_learning_rule(self, x, y):
 
        for class_index in range(10):
 
            # Turning the multi-class label into a binary target: 1 for the perceptron matching the true class, 0 for the rest
            if class_index == y:
                target = 1
            else:
                target = 0
 
            self.perceptrons[class_index].apply_learning_rule(
                x, target
            )
 

In [19]:
# TASK 3 
#Tracking final validation accuracy for each config so I can compare them

#Grayscale: fixed-epoch runs across different learning rates 

np.random.seed(1234)
p_gray = MultiClassPerceptron(
    n_inputs=X_train_gray.shape[1],
    n_classes=10,
    alpha=0.1
)

train(
    p_gray,
    X_train_gray,
    y_train_gray,
    X_valid_gray,
    y_valid_gray,
    epochs=20
)

Epoch 1 - Validation accuracy: 0.17
Epoch 2 - Validation accuracy: 0.14
Epoch 3 - Validation accuracy: 0.15
Epoch 4 - Validation accuracy: 0.22
Epoch 5 - Validation accuracy: 0.22
Epoch 6 - Validation accuracy: 0.23
Epoch 7 - Validation accuracy: 0.21
Epoch 8 - Validation accuracy: 0.15
Epoch 9 - Validation accuracy: 0.16
Epoch 10 - Validation accuracy: 0.20
Epoch 11 - Validation accuracy: 0.24
Epoch 12 - Validation accuracy: 0.20
Epoch 13 - Validation accuracy: 0.13
Epoch 14 - Validation accuracy: 0.15
Epoch 15 - Validation accuracy: 0.26
Epoch 16 - Validation accuracy: 0.23
Epoch 17 - Validation accuracy: 0.21
Epoch 18 - Validation accuracy: 0.16
Epoch 19 - Validation accuracy: 0.23
Epoch 20 - Validation accuracy: 0.21


In [20]:
np.random.seed(1234)
p_gray = MultiClassPerceptron(
    n_inputs=X_train_gray.shape[1],
    n_classes=10,
    alpha=0.01
)

train(
    p_gray,
    X_train_gray,
    y_train_gray,
    X_valid_gray,
    y_valid_gray,
    epochs=20
)


Epoch 1 - Validation accuracy: 0.14
Epoch 2 - Validation accuracy: 0.19
Epoch 3 - Validation accuracy: 0.15
Epoch 4 - Validation accuracy: 0.24
Epoch 5 - Validation accuracy: 0.13
Epoch 6 - Validation accuracy: 0.21
Epoch 7 - Validation accuracy: 0.20
Epoch 8 - Validation accuracy: 0.23
Epoch 9 - Validation accuracy: 0.19
Epoch 10 - Validation accuracy: 0.23
Epoch 11 - Validation accuracy: 0.21
Epoch 12 - Validation accuracy: 0.23
Epoch 13 - Validation accuracy: 0.17
Epoch 14 - Validation accuracy: 0.16
Epoch 15 - Validation accuracy: 0.20
Epoch 16 - Validation accuracy: 0.20
Epoch 17 - Validation accuracy: 0.22
Epoch 18 - Validation accuracy: 0.21
Epoch 19 - Validation accuracy: 0.16
Epoch 20 - Validation accuracy: 0.19


In [21]:
np.random.seed(1234)
p_gray = MultiClassPerceptron(
    n_inputs=X_train_gray.shape[1],
    n_classes=10,
    alpha=0.001
)

train(
    p_gray,
    X_train_gray,
    y_train_gray,
    X_valid_gray,
    y_valid_gray,
    epochs=20
)

Epoch 1 - Validation accuracy: 0.14
Epoch 2 - Validation accuracy: 0.14
Epoch 3 - Validation accuracy: 0.16
Epoch 4 - Validation accuracy: 0.22
Epoch 5 - Validation accuracy: 0.15
Epoch 6 - Validation accuracy: 0.20
Epoch 7 - Validation accuracy: 0.21
Epoch 8 - Validation accuracy: 0.22
Epoch 9 - Validation accuracy: 0.19
Epoch 10 - Validation accuracy: 0.21
Epoch 11 - Validation accuracy: 0.26
Epoch 12 - Validation accuracy: 0.18
Epoch 13 - Validation accuracy: 0.18
Epoch 14 - Validation accuracy: 0.19
Epoch 15 - Validation accuracy: 0.23
Epoch 16 - Validation accuracy: 0.24
Epoch 17 - Validation accuracy: 0.20
Epoch 18 - Validation accuracy: 0.24
Epoch 19 - Validation accuracy: 0.19
Epoch 20 - Validation accuracy: 0.17


In [22]:
#RGB: fixed-epoch runs across different learning rates 
np.random.seed(1234)
p_rgb = MultiClassPerceptron(
    n_inputs=X_train_rgb.shape[1],
    n_classes=10,
    alpha=0.1
)

train(
    p_rgb,
    X_train_rgb,
    y_train_rgb,
    X_valid_rgb,
    y_valid_rgb,
    epochs=20
)


Epoch 1 - Validation accuracy: 0.26
Epoch 2 - Validation accuracy: 0.22
Epoch 3 - Validation accuracy: 0.27
Epoch 4 - Validation accuracy: 0.36
Epoch 5 - Validation accuracy: 0.33
Epoch 6 - Validation accuracy: 0.35
Epoch 7 - Validation accuracy: 0.33
Epoch 8 - Validation accuracy: 0.33
Epoch 9 - Validation accuracy: 0.37
Epoch 10 - Validation accuracy: 0.26
Epoch 11 - Validation accuracy: 0.34
Epoch 12 - Validation accuracy: 0.38
Epoch 13 - Validation accuracy: 0.37
Epoch 14 - Validation accuracy: 0.36
Epoch 15 - Validation accuracy: 0.38
Epoch 16 - Validation accuracy: 0.37
Epoch 17 - Validation accuracy: 0.31
Epoch 18 - Validation accuracy: 0.33
Epoch 19 - Validation accuracy: 0.36
Epoch 20 - Validation accuracy: 0.37


In [23]:
np.random.seed(1234)
p_rgb = MultiClassPerceptron(
    n_inputs=X_train_rgb.shape[1],
    n_classes=10,
    alpha=0.01
)

train(
    p_rgb,
    X_train_rgb,
    y_train_rgb,
    X_valid_rgb,
    y_valid_rgb,
    epochs=20
)

Epoch 1 - Validation accuracy: 0.25
Epoch 2 - Validation accuracy: 0.35
Epoch 3 - Validation accuracy: 0.39
Epoch 4 - Validation accuracy: 0.36
Epoch 5 - Validation accuracy: 0.26
Epoch 6 - Validation accuracy: 0.31
Epoch 7 - Validation accuracy: 0.38
Epoch 8 - Validation accuracy: 0.33
Epoch 9 - Validation accuracy: 0.37
Epoch 10 - Validation accuracy: 0.36
Epoch 11 - Validation accuracy: 0.35
Epoch 12 - Validation accuracy: 0.40
Epoch 13 - Validation accuracy: 0.39
Epoch 14 - Validation accuracy: 0.35
Epoch 15 - Validation accuracy: 0.38
Epoch 16 - Validation accuracy: 0.42
Epoch 17 - Validation accuracy: 0.32
Epoch 18 - Validation accuracy: 0.27
Epoch 19 - Validation accuracy: 0.41
Epoch 20 - Validation accuracy: 0.33


In [25]:
np.random.seed(1234)
p_rgb = MultiClassPerceptron(
    n_inputs=X_train_rgb.shape[1],
    n_classes=10,
    alpha=0.001
)

train(
    p_rgb,
    X_train_rgb,
    y_train_rgb,
    X_valid_rgb,
    y_valid_rgb,
    epochs=20 
)

Epoch 1 - Validation accuracy: 0.21
Epoch 2 - Validation accuracy: 0.24
Epoch 3 - Validation accuracy: 0.30
Epoch 4 - Validation accuracy: 0.38
Epoch 5 - Validation accuracy: 0.29
Epoch 6 - Validation accuracy: 0.36
Epoch 7 - Validation accuracy: 0.32
Epoch 8 - Validation accuracy: 0.32
Epoch 9 - Validation accuracy: 0.32
Epoch 10 - Validation accuracy: 0.30
Epoch 11 - Validation accuracy: 0.34
Epoch 12 - Validation accuracy: 0.38
Epoch 13 - Validation accuracy: 0.43
Epoch 14 - Validation accuracy: 0.37
Epoch 15 - Validation accuracy: 0.39
Epoch 16 - Validation accuracy: 0.36
Epoch 17 - Validation accuracy: 0.36
Epoch 18 - Validation accuracy: 0.30
Epoch 19 - Validation accuracy: 0.36
Epoch 20 - Validation accuracy: 0.38


In [26]:
def train_early_stopping(model, X_train, y_train, X_valid, y_valid,
                          max_epochs=20, patience=3):
 
    best_accuracy = 0.0
    epochs_without_improvement = 0
    # Storing a copy of the weights/bias whenever validation accuracy improves,
    # so I can restore the best version instead of whatever the model looks like when training stops
    best_weights = [np.copy(p.weights) for p in model.perceptrons]
    best_biases = [p.bias for p in model.perceptrons]
 
    for epoch in range(max_epochs):
 
        indices = np.random.permutation(len(X_train))
 
        for i in indices:
            model.apply_learning_rule(X_train[i], y_train[i])
 
        correct = 0
        for i in range(len(X_valid)):
            if model.predict(X_valid[i]) == y_valid[i]:
                correct += 1
 
        accuracy = correct / len(X_valid)
 
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            epochs_without_improvement = 0
            best_weights = [np.copy(p.weights) for p in model.perceptrons]
            best_biases = [p.bias for p in model.perceptrons]
        else:
            epochs_without_improvement += 1
 
        if epochs_without_improvement >= patience:
            break
 
    # Restoring the model to its best-performing epoch
    for p, w, b in zip(model.perceptrons, best_weights, best_biases):
        p.weights = w
        p.bias = b
 
    return best_accuracy

In [27]:
# Creating a new grayscale multi-class perceptron with alpha=0.01 - learning rate that performed best in earlier sweep
np.random.seed(1234)
p_gray_early = MultiClassPerceptron(
    n_inputs=X_train_gray.shape[1],
    n_classes=10,
    alpha=0.01
)
 
results["gray_early_stop"] = train_early_stopping(
    p_gray_early,
    X_train_gray,
    y_train_gray,
    X_valid_gray,
    y_valid_gray,
    max_epochs=20,
    patience=3
)
 

Stopped early at epoch 7


In [28]:
# Same idea but for the RGB model, using alpha=0
np.random.seed(1234)
p_rgb_early = MultiClassPerceptron(
    n_inputs=X_train_rgb.shape[1],
    n_classes=10,
    alpha=0.1
)
 
results["rgb_early_stop"] = train_early_stopping(
    p_rgb_early,
    X_train_rgb,
    y_train_rgb,
    X_valid_rgb,
    y_valid_rgb,
    max_epochs=20,
    patience=3
)
 

Stopped early at epoch 7
